# BTW Module 03: Publication-Grade Visualizations
**Downstream Bulk Transcriptomics Workbench (`btw`)**

สมุดงานตัวอย่างสาธิตการใช้งาน **FR-4 (Visualization Module)**:
1. การกำหนดสไตล์ระดับวารสารวิชาการชั้นนำ (Nature, Cell, Science) ผ่าน `set_publication_style`
2. **Volcano Plot:** แสดงยีน UP / DOWN พร้อมติดฉลากยีนสำคัญแบบไม่ทับซ้อนด้วย `adjustText` และโหมด Interactive Plotly
3. **PCA Plot:** แสดง Variance Explained (%) และจัดกลุ่มตัวอย่างตาม metadata (2D & 3D)
4. **MA Plot:** แสดง Mean Expression vs log2FC พร้อมเน้นยีนที่มีนัยสำคัญ
5. **Dispersion Plot:** แสดงการประมาณค่า Dispersions จาก PyDESeq2
6. **Hierarchical Clustered Heatmap:** แสดง Top DEGs พร้อมแถบสีจำแนกกลุ่มตัวอย่างตาม metadata
7. **Cross-Contrast Overlaps:** เปรียบเทียบยีนร่วมข้าม Contrast ด้วย UpSet Plot และ Venn Diagram
8. การบันทึกรูปภาพความละเอียดสูง (PNG 300+ DPI, SVG, PDF) สำหรับตีพิมพ์

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import btw
from btw import set_seed, logger
from btw.de_analysis import run_de, run_multiple_contrasts
from btw.viz import (
    set_publication_style,
    plot_volcano,
    plot_pca,
    plot_ma,
    plot_dispersion,
    plot_heatmap,
    plot_upset,
    plot_venn,
    save_figure,
)

# กำหนดธีมสไตล์ภาพและ seed
set_seed(42)
set_publication_style(palette="nature", dpi=150)
print(f"BTW version: {btw.__version__}")

## 1. เตรียมข้อมูลและการวิเคราะห์ Differential Expression
สร้าง Synthetic Bulk RNA-seq count matrix และรัน DE Analysis เพื่อนำผลลัพธ์มาพล็อต

In [ ]:
genes = [f"GENE_{i:03d}" for i in range(1, 101)]
samples = ["ctrl_1", "ctrl_2", "ctrl_3", "treat_1", "treat_2", "treat_3"]

np.random.seed(42)
base_counts = np.random.negative_binomial(5, 0.01, size=(100, 6))
base_counts[:15, 3:] = (base_counts[:15, 3:] * 4.5).astype(int)
base_counts[15:30, 3:] = (base_counts[15:30, 3:] * 0.2).astype(int)
counts_df = pd.DataFrame(np.clip(base_counts, 0, None), index=genes, columns=samples)

metadata_df = pd.DataFrame({
    "sample_id": samples,
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch2", "batch1", "batch2", "batch1", "batch2"],
}).set_index("sample_id")

# รัน DE Analysis
de_res = run_de(counts_df, metadata_df, contrast=("condition", "treated", "control"))
print(de_res.summary())

## 2. Volcano Plot (FR-4)
สร้าง Volcano plot พร้อม label ยีนสำคัญ 10 อันดับแรกโดยใช้ `adjustText` ป้องกันการทับซ้อน

In [ ]:
fig_volcano, ax = plot_volcano(
    de_res,
    padj_cutoff=0.05,
    lfc_cutoff=1.0,
    top_n_labels=10,
    interactive=False,
    title="Volcano Plot: Treated vs Control (adjustText labels)",
)
plt.show()

### Interactive Volcano Plot (Plotly Mode)
สร้างกราฟแบบ interactive ที่สามารถ hover ดูข้อมูล log2FC, -log10(padj), และ Gene ID

In [ ]:
plotly_volcano = plot_volcano(
    de_res,
    padj_cutoff=0.05,
    lfc_cutoff=1.0,
    interactive=True,
)
# plotly_volcano.show() # ใน JupyterLab สามารถเรียก .show() เพื่อดู interactive chart ได้โดยตรง
print(f"Created interactive Plotly Volcano chart with {len(plotly_volcano.data)} data traces.")

## 3. PCA Plot (FR-4)
คำนวณ PCA จาก scikit-learn และพล็อตการจัดกลุ่มตัวอย่างตาม `condition` และ `batch`

In [ ]:
fig_pca, ax_pca, pca_coords = plot_pca(
    data=counts_df,
    metadata=metadata_df,
    color_by="condition",
    shape_by="batch",
    top_n_variable_genes=100,
    show_sample_labels=True,
    interactive=False,
)
plt.show()
display(pca_coords)

## 4. MA Plot (FR-4)
พล็อตความสัมพันธ์ระหว่าง Mean Expression และ log2 Fold Change

In [ ]:
fig_ma, ax_ma = plot_ma(
    de_res,
    padj_cutoff=0.05,
    lfc_cutoff=1.0,
    interactive=False,
)
plt.show()

## 5. Dispersion Plot (FR-4)
แสดงผลการประมาณค่า Dispersions (Gene-wise, Fitted Trend, MAP Shrinkage) จาก PyDESeq2

In [ ]:
fig_disp, ax_disp = plot_dispersion(de_res)
plt.show()

## 6. Hierarchical Clustered Heatmap (FR-4)
พล็อต Heatmap ของ Top DEGs พร้อมแถบสี metadata จำแนกกลุ่มตัวอย่าง

In [ ]:
g_heatmap = plot_heatmap(
    counts_df,
    metadata=metadata_df,
    de_result=de_res,
    top_n_degs=30,
    annotation_cols=["condition", "batch"],
    z_score=True,
    title="Hierarchical Clustering of Top DEGs",
)
plt.show()

## 7. Cross-Contrast Overlaps: UpSet Plot & Venn Diagram (FR-4)

In [ ]:
contrasts_list = [
    ("condition", "treated", "control"),
    ("batch", "batch2", "batch1"),
]
multi_res = run_multiple_contrasts(counts_df, metadata_df, contrasts=contrasts_list)

# UpSet Plot
axes_upset = plot_upset(
    multi_res,
    title="Cross-Contrast DEG Intersections (UpSet Plot)",
)
plt.show()

# Venn Diagram
fig_venn, ax_venn = plot_venn(
    multi_res,
    title="DEG Overlaps (Venn Summary)",
)
plt.show()

## 8. การบันทึกภาพความละเอียดสูง (Publication-grade Figure Export)
บันทึกไฟล์ภาพในฟอร์แมต PNG (300 DPI) และเวกเตอร์ SVG

In [ ]:
out_dir = Path("results/example_viz")
out_dir.mkdir(parents=True, exist_ok=True)

png_file = out_dir / "volcano_plot_publication.png"
svg_file = out_dir / "volcano_plot_publication.svg"

save_figure(fig_volcano, png_file, dpi=300)
save_figure(fig_volcano, svg_file)

print(f"Saved high-res PNG to: {png_file.resolve()}")
print(f"Saved vector SVG to:   {svg_file.resolve()}")